# Phase 0 — nerfstudio `splatfacto` on a free Colab GPU

Runs the **"running nerfstudio"** steps from the video on a cloud **NVIDIA T4**, because this project's local box is an **AMD RX 7600** — `splatfacto` training is CUDA-only, so it can't run there (see [`../TODO.md`](../TODO.md) GPU note and [`../cloud/README.md`](../cloud/README.md)).

**Video step → Colab cell**

| Video step | Here |
|---|---|
| Install nerfstudio (the *separate* install video) | Cell **1** — `pip install nerfstudio` |
| `conda activate nerfstudio` + `cd nerfstudio` | Not needed — Colab is one global env |
| `ns-download-data nerfstudio` (the *poster* set) | Cell **2** |
| Train your first splat | Cell **3** — `ns-train splatfacto` |
| — (get the trained `.ply` out) | Cell **4** — `ns-export gaussian-splat` |

**Run order:** *Runtime ▸ Change runtime type ▸* **T4 GPU**, then run every cell top to bottom.


## 0 · Confirm you're on a GPU runtime
If `nvidia-smi` errors, set *Runtime ▸ Change runtime type ▸* **GPU (T4)** and re-run this cell.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu available?', torch.cuda.is_available())

## 1 · Install nerfstudio  (= the "separate install video")

For **splatfacto** we only need `gsplat` (CUDA), **not** `tiny-cuda-nn` (that's only for NeRF methods like `nerfacto`) — so we skip the most fragile part of a nerfstudio install. `pip install nerfstudio` pulls `gsplat` in automatically; its CUDA kernels JIT-compile on first training run, so give the first `ns-train` a few extra minutes.

The last line prints the `ns-train` path on success. If it says **MISSING**, the install failed — read the pip error just above it (nerfstudio versions move fast; see **Troubleshooting** at the bottom).

In [ ]:
!pip install --upgrade pip -q
# NOT quiet, on purpose: if the install fails, the error must be visible.
!pip install nerfstudio 2>&1 | tail -n 50
print('----- verify: a path printed below = success -----')
!which ns-train || echo 'ns-train MISSING - read the pip error above (often a dep like open3d/pymeshlab with no wheel for this Python). Paste the last ~15 lines for a pinned fix.'

## 1b · Compile gsplat's CUDA kernels — once per session (~20–35 min, be patient)

There is **no prebuilt gsplat wheel** for Colab's torch/CUDA combo (checked 2026-07: the wheel index tops out at torch 2.4/cu124, Colab ships 2.11/cu128), so gsplat must JIT-compile its 26 CUDA kernels. Left to its default of **10 parallel nvcc jobs** it exhausts Colab's ~12.7 GB RAM and the OOM killer terminates the build with `Killed` / exit 137 — mid-training, cryptically. `MAX_JOBS=2` trades speed for staying alive.

Run this **before** any `ns-train`. It looks stuck for long stretches — that's nvcc working. If it ever does die partway, just re-run it: ninja reuses every object that already built.

In [ ]:
# One-time per session. Importing gsplat's backend triggers the JIT build.
# MAX_JOBS=2 keeps peak RAM under Colab's ~12.7 GB so the OOM killer stays away.
!MAX_JOBS=2 python -c "from gsplat.cuda._backend import _C; print('gsplat CUDA backend ready:', _C is not None)"

### (optional) Persist outputs to Google Drive
Colab sessions are ephemeral. Mount Drive so the trained `.ply` survives a disconnect. Un-comment to use.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive

## 2 · Download the sample data  (`ns-download-data nerfstudio`)
The **poster** capture: low-res images + COLMAP camera poses (`transforms.json`). `--capture-name poster` grabs just that one; the bare `ns-download-data nerfstudio` from the video pulls *every* capture (much larger).

> ⚠️ **Known failure (2026-07):** nerfstudio hosts this on Google Drive and the download regularly dies with a `gdown ... Cannot retrieve the public link` quota error. That's their hosting, not your setup. If it happens, **skip §2–§3 and go straight to §5** — training on your own capture doesn't touch Drive hosting at all.

In [ ]:
!ns-download-data nerfstudio --capture-name poster
!ls -la data/nerfstudio/poster

## 3 · Train your first splat  (`ns-train splatfacto`)
Headless (no live viewer) with TensorBoard logging. `--max-num-iterations 7000` gives a quick first result (~10–20 min on a T4); the default is 30000 — raise it for quality. Output lands in `outputs/poster/splatfacto/<timestamp>/`.

In [ ]:
# Run §1b (gsplat compile) first; MAX_JOBS=2 is just a safety net if you skipped it.
!MAX_JOBS=2 ns-train splatfacto --data data/nerfstudio/poster --max-num-iterations 7000 --vis tensorboard

## 4 · Export the trained `.ply`  (the Gaussians)
`ns-export gaussian-splat` writes `splat.ply` — thousands of 3D Gaussians (xyz, scale, rotation, color, opacity). This is exactly the file your **Phase 1 Three.js viewer** parses. Download it and drop it into [`../scenes/`](../scenes/) locally.

This cell auto-finds the **most recent** `splatfacto` run — the poster sample, or your own capture from §5 below, whichever you trained last.

In [ ]:
import glob, os
cfgs = sorted(glob.glob('outputs/*/splatfacto/*/config.yml'), key=os.path.getmtime)  # poster OR your own run - whichever trained last
assert cfgs, 'No config.yml found - did training start at all?'
CONFIG = cfgs[-1]
# config.yml is written at training START. Only a checkpoint proves training ran:
ckpts = glob.glob(os.path.join(os.path.dirname(CONFIG), 'nerfstudio_models', '*.ckpt'))
assert ckpts, ('No checkpoint next to ' + CONFIG + ' - training did NOT complete. '
               'Re-run the training cell and let it reach the final iteration '
               '(first run stalls ~3-5 min while gsplat JIT-compiles - that is normal).')
print('using config    :', CONFIG)
print('using checkpoint:', ckpts[-1])
!ns-export gaussian-splat --load-config {CONFIG} --output-dir exports/splat
!ls -la exports/splat

In [ ]:
import glob
plys = glob.glob('exports/splat/*.ply')
assert plys, 'No .ply in exports/splat - the export cell above must succeed first.'
from google.colab import files
files.download(plys[0])  # (or copy to Drive if you mounted it above)

## 5 · (Your own capture) Train on the Iceland COLMAP output

Train on **your own** scene, reusing the COLMAP solve the `phase0-capture` `.bat` already produced — **no COLMAP re-run, no `ns-process-data`**.

**Why not `ns-process-data --skip-colmap`?** It refuses to start without a COLMAP binary installed (`Could not find COLMAP`), even when skipping it. Instead we hand `ns-train` the COLMAP model **directly** via its built-in `colmap` dataparser, which reads `images/` + `sparse/0/*.bin` as-is.

**One-time upload:** a clean, upload-ready zip was staged locally at **`Desktop\iceland_upload.zip`** (40 MB). It contains only what's needed — the 405 frames + `sparse/0`, the **largest** of the three fragmented sub-models (140 registered images vs 92 / 82; FPV footage fragments, see `phase0-capture/README.md` gotchas). Upload it to Drive at **`MyDrive/proj/iceland_upload.zip`**, then run the cells below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip from Drive to the Colab VM's local disk (much faster I/O for training than reading off Drive).
!unzip -q -n "/content/drive/MyDrive/proj/iceland_upload.zip" -d /content/
print('----- expect images/ (405 jpgs) and sparse/0/*.bin -----')
!ls /content/iceland
!ls /content/iceland/sparse/0
!ls /content/iceland/images | wc -l

In [ ]:
# Train straight off the COLMAP model - the trailing `colmap` selects nerfstudio's
# ColmapDataParser (no transforms.json needed). Dataparser flags go AFTER `colmap`.
# NOTE: only the 140 frames registered in sparse/0 are used - that's correct, the
# other frames belong to the two smaller disconnected sub-models.
#
# Run §1b (gsplat compile) FIRST. MAX_JOBS=2 here is only a safety net in case you
# skipped it. LET TRAINING RUN TO THE END (~15-25 min on a T4): checkpoints save
# every 2000 steps + at the final iteration; the export cell refuses to run
# without one. Wait for 7000/7000 before moving on.
!MAX_JOBS=2 ns-train splatfacto --data /content/iceland \
    --max-num-iterations 7000 --vis tensorboard \
    colmap --colmap-path sparse/0 --images-path images
# If `colmap` is rejected as a subcommand on a future nerfstudio version, list valid
# dataparsers with:  !ns-train splatfacto --help   (look for the trailing positional)

## Troubleshooting
- **Build dies with `Killed` / exit code 137 during `gsplat: Setting up CUDA`:** that's the Linux **OOM killer** — gsplat defaults to 10 parallel nvcc jobs and each needs 1.5–3 GB RAM; Colab's free VM has ~12.7 GB. Fix: compile with `MAX_JOBS=2` (§1b). Ninja resumes — already-built objects are reused, so a retry never starts from zero.
- **No prebuilt gsplat wheel:** checked 2026-07 — `docs.gsplat.studio/whl` tops out at torch 2.4 / cu124; Colab runs torch 2.11 / cu128, so JIT compile is unavoidable. Re-check that index if Colab feels slow — a matching wheel would skip §1b entirely.
- **§2 poster download fails with a `gdown` quota error:** known nerfstudio hosting issue (Google Drive download cap) — nothing to fix on your side. Skip to §5 and train on your own capture.
- **`ns-process-data` says `Could not find COLMAP`:** it requires the COLMAP binary even with `--skip-colmap`. We avoid it entirely (§5 uses the `colmap` dataparser). If you ever *do* need it on Colab: `!apt-get install -y colmap -q`.
- **`pip install nerfstudio` changed torch and broke CUDA:** reinstall Colab's torch afterward, matching the CUDA shown by `nvidia-smi`, e.g. `!pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu121`.
- **Want the live viewer?** nerfstudio's viewer needs a public tunnel on Colab. Easiest is to skip it (we use `--vis tensorboard`) and inspect curves with `%load_ext tensorboard` then `%tensorboard --logdir outputs`.
- **Version drift:** nerfstudio moves fast (their docs say so). If an `ns-*` flag is rejected, check `!ns-train splatfacto --help` and `!ns-export gaussian-splat --help`.
